# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

# Values to edit

In [2]:
max_cloud_cover = 5 # percentage
low_tide_delta = 1 # hours
lowtide_search_range=60

In [3]:
site_names = ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [4]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:59968/status,
Dashboard: http://127.0.0.1:59968/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59970,Workers: 4
Dashboard: http://127.0.0.1:59968/status,Total threads: 8
Started: Just now,Total memory: 31.73 GiB
Comm: tcp://127.0.0.1:59992,Total threads: 2
Dashboard: http://127.0.0.1:59993/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:59973,


In [5]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
lowtide_search_range_file = data_path / "ELF24505_satellite_day_search_range.csv"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"

In [7]:
for site_name in site_names:
    sampling.get_site_satellite(
        site_name=site_name,
        survey_dates_file=servey_dates_file,
        lowtide_search_range=lowtide_search_range,
        low_tide_delta=low_tide_delta,
        uav_folder=uav_folder,
        max_cloud_cover=max_cloud_cover,
    )

Site CatlinsLake
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2022-07-11/2022-09-09
	Time from low tide: 0:07:19.024000 for S2B_MSIL2A_20220715T223719_R072_T59GLJ_20220716T080759
	Time from low tide: 0:19:21.024000 for S2A_MSIL2A_20220730T223721_R072_T59GLJ_20220731T212610
	Time from low tide: 0:11:21.024000 for S2A_MSIL2A_20220829T223721_R072_T59GLJ_20240720T123310
	Time from low tide: 0:11:21.024000 for S2A_MSIL2A_20220829T223721_R072_T59GLJ_20220831T082405
	Time from low tide: 0:22:50.976000 for S2B_MSIL2A_20220814T223709_R072_T59GLJ_20220815T075048
	Low tide tiles: 4 from a total lowish cloud cover tiles of 34
	No cloud tiles: 1 from the low tide tiles of 4. Cloud percentages [ 0.  10.2  8.9 10.2] Kept [ True False False False]
Site CatlinsRiverMouth
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2023-05-08/2023-07-07
	Time from low tide: 0:11:50.976000 for S2B_MSIL2A_20230620T223709_R072_T59GLJ_20240813T

In [9]:
for site_name in site_names:
    print(f"Save RGB for {site_name}")
    satellite_images_path = utils.get_satellite_path(site_name=site_name, low_tide_delta=low_tide_delta, max_cloud_cover=max_cloud_cover)
    satellite_data = utils.load_satellite(filename=satellite_images_path)
    rgb_folder = satellite_images_path.parent / "rgb"
    rgb_folder.mkdir(exist_ok=True)
    for time_index in range(satellite_data.sizes["time"]):
        rgb = satellite_data[["B04", "B03", "B02"]].isel(time=time_index).to_array("band")
        date = str(satellite_data.time.isel(time=time_index).values)[:10]

        # Sentinel-2 reflectance is scaled to 0-10000; create an 8-bit display RGB.
        rgb = ((rgb.fillna(0).clip(min=0, max=3000) / 3000 * 255)
               .round()
               .astype("uint8")
               .rio.write_nodata(0))
        rgb.rio.to_raster(
            rgb_folder / f"{satellite_images_path.stem}_{date}.tif",
            dtype="uint8",
            photometric="RGB",
            compress="ZSTD",
        )

Save RGB for CatlinsLake
Save RGB for CatlinsRiverMouth
Save RGB for Childrens
Save RGB for Duvauchelle
Save RGB for Robinsons
Save RGB for Takamatua
Save RGB for Purau
Save RGB for Ihutai
Save RGB for IveyBay_Nov25
Save RGB for IveyBay_Feb26
Save RGB for LeftBank_Nov25
Save RGB for LeftBank_Feb26
Save RGB for Paremata_Nov25
Save RGB for Paremata_Feb26
Save RGB for Paremata_Feb25
Save RGB for ThePoint_Nov25
Save RGB for ThePoint_Feb26
Save RGB for Takapuwahia_Nov25
Save RGB for Takapuwahia_Feb26
Save RGB for IveyBay_ThePoint_LeftBank_Oct24
